# Tutorial 16: Transfer learning across GeneralizedNCap geometry regimes

Tutorial 15 separated two simulation campaigns. Here we ask a stricter geometry question:

> Can a v0 foundation model trained on one finger-count regime become a strong specialist
> in a different regime with only a fraction of that regime's labels?

We partition **all 3,683 GeneralizedCapNInterdigital designs only by geometry**:
`2-4`, `5-7`, and `8-10` fingers. These domains are almost perfectly balanced.

## Learning goals

1. Build reproducible geometry domains without using campaign labels.
2. Compare zero-shot, same-label target-only, transfer, and full dedicated models.
3. Read learning curves using macro R2, MAPE, and strict within-5% accuracy.
4. Separate two claims: transfer helps in the few-label regime; enough target data still matters.

In [1]:
import json
import logging
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots

from squadds.layouts import (
    StaticEmbeddingClient,
    V0PartitionTransferStudy,
    V0TransferLearningStudy,
    canonical_design_id,
    regression_scores,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 20)
logging.getLogger("httpx").setLevel(logging.WARNING)

EMBEDDING_REVISION = os.getenv("SQUADDS_EMBEDDING_REVISION", "main")
DATABASE_REVISION = os.getenv("SQUADDS_DATABASE_REVISION", "main")
TARGET_NAMES = ["C(N,S)", "C(N,G)", "C(S,G)"]
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75]
DOMAIN_COLORS = {
    "2-4 fingers": "#D1495B",
    "5-7 fingers": "#E9C46A",
    "8-10 fingers": "#00798C",
}
METHOD_COLORS = {
    "zero-shot": "#6A4C93",
    "target-only": "#D1495B",
    "transfer": "#00798C",
    "dedicated-full": "#1F2937",
}

## 1. Load and partition by an actual class option

`finger_count` is a meaningful option of `GeneralizedCapNInterdigital`, not a campaign
identifier or a learned cluster. Grouping three adjacent values per domain produces a
fair split while making extrapolation progressively harder.

In [2]:
embedding_client = StaticEmbeddingClient(revision=EMBEDDING_REVISION)
embeddings = embedding_client.embeddings().loc[
    lambda frame: frame["component_name"] == "GeneralizedCapNInterdigital"
]
database_path = hf_hub_download(
    "SQuADDS/SQuADDS_DB",
    "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
    repo_type="dataset",
    revision=DATABASE_REVISION,
)
with open(database_path, encoding="utf-8") as stream:
    simulation_rows = json.load(stream)


def parse_um(value):
    return float(str(value).replace("um", ""))


def finger_domain(count):
    if count <= 4:
        return "2-4 fingers"
    if count <= 7:
        return "5-7 fingers"
    return "8-10 fingers"


records = []
for row in simulation_rows:
    options = row["design"]["design_options"]
    results = row["sim_results"]
    count = int(options["finger_count"])
    records.append(
        {
            "design_id": canonical_design_id("GeneralizedCapNInterdigital", options),
            "domain": finger_domain(count),
            "finger_count": count,
            "finger_length_um": parse_um(options["finger_length"]),
            "finger_width_um": parse_um(options["finger_width"]),
            "finger_gap_um": parse_um(options["finger_gap_north_south"]),
            "C(N,S)": results["north_to_south"],
            "C(N,G)": results["north_to_ground"],
            "C(S,G)": results["south_to_ground"],
        }
    )

data = embeddings.merge(pd.DataFrame(records), on="design_id", validate="one_to_one")
domain_counts = data["domain"].value_counts().reindex(DOMAIN_COLORS)
domain_counts.to_frame("designs")

,designs
domain,
2-4 fingers,1230
5-7 fingers,1224
8-10 fingers,1229


In [3]:
# %% hide input
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Balanced geometry domains", "What each domain occupies"],
    column_widths=[0.32, 0.68],
)
fig.add_trace(
    go.Bar(
        x=domain_counts.index,
        y=domain_counts.values,
        marker_color=[DOMAIN_COLORS[name] for name in domain_counts.index],
        text=domain_counts.values,
        textposition="outside",
        hovertemplate="%{x}<br>%{y:,} designs<extra></extra>",
    ),
    row=1,
    col=1,
)
for domain in DOMAIN_COLORS:
    frame = data.loc[data["domain"] == domain]
    fig.add_trace(
        go.Scattergl(
            x=frame["finger_count"],
            y=frame["finger_length_um"],
            mode="markers",
            name=domain,
            marker={"size": 6, "opacity": 0.45, "color": DOMAIN_COLORS[domain]},
            customdata=np.column_stack([frame["finger_width_um"], frame["finger_gap_um"]]),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>fingers=%{x}<br>length=%{y} um"
                "<br>width=%{customdata[0]} um<br>gap=%{customdata[1]} um<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="geometry domain", row=1, col=1)
fig.update_yaxes(title_text="designs", row=1, col=1, range=[0, 1400])
fig.update_xaxes(title_text="finger count", row=1, col=2)
fig.update_yaxes(title_text="finger length (um)", row=1, col=2)
fig.update_layout(
    title="A class-native partition, independent of simulation campaign",
    height=520,
    template="plotly_white",
)
fig.show()

## 2. Define the foundation and specialist protocol

The largest domain, `2-4 fingers`, is the foundation source. For each target domain:

1. Hold out 30% once for testing.
2. Draw 5%, 10%, 25%, 50%, or 75% from the remaining adaptation pool.
3. Fit a target-only head and a transfer head regularized toward the source head.
4. Compare both with zero-shot and a dedicated model trained on the complete pool.

The v0 projector is fitted **only on source embeddings**. Target test rows never set
normalization, model weights, or hyperparameters.

In [4]:
embedding_matrix = np.vstack(data["embedding"]).astype(np.float32)
target_matrix = data[TARGET_NAMES].to_numpy(dtype=float)
study = V0PartitionTransferStudy(
    embedding_matrix,
    target_matrix,
    data["domain"].to_numpy(),
    "2-4 fingers",
    target_names=TARGET_NAMES,
    pooled_shape_size=12,
    alpha=10.0,
)

pd.concat(
    [
        study.domain_counts(),
        study.domain_similarity().rename(columns={"target_domain": "domain"}),
    ],
    ignore_index=True,
    sort=False,
)

,domain,rows,role,source_domain,minimum,median,mean,maximum
0,2-4 fingers,1230.0,source,NaN,NaN,NaN,NaN,NaN
1,5-7 fingers,1224.0,target,NaN,NaN,NaN,NaN,NaN
2,8-10 fingers,1229.0,target,NaN,NaN,NaN,NaN,NaN
3,5-7 fingers,NaN,NaN,2-4 fingers,0.216200,0.545620,0.594068,0.942800
4,8-10 fingers,NaN,NaN,2-4 fingers,0.087159,0.372537,0.368341,0.780595


In [5]:
# %% hide input
fig = go.Figure(
    go.Sankey(
        node={
            "label": [
                "foundation: 2-4 fingers",
                "5-7 adaptation: 5-75%",
                "5-7 held-out: 30%",
                "8-10 adaptation: 5-75%",
                "8-10 held-out: 30%",
            ],
            "color": ["#D1495B", "#E9C46A", "#F4DFA0", "#00798C", "#8CCBD4"],
            "pad": 22,
            "thickness": 22,
        },
        link={
            "source": [0, 0, 0, 0],
            "target": [1, 2, 3, 4],
            "value": [70, 30, 70, 30],
            "color": ["rgba(233,196,106,.35)", "rgba(233,196,106,.15)", "rgba(0,121,140,.35)", "rgba(0,121,140,.15)"],
        },
    )
)
fig.update_layout(
    title="One source prior, two independently held-out target regimes",
    height=430,
    template="plotly_white",
)
fig.show()

## 3. Run percentage-based learning curves

R2 measures explained variance; MAPE keeps errors interpretable in percent; and
within-5% accuracy asks how often an individual capacitance prediction is very close.
Twelve repeated label draws expose sampling variability.

In [6]:
curves = study.learning_curves(
    FRACTIONS,
    repeats=12,
    test_fraction=0.30,
    random_seed=16,
)
benchmarks = study.dedicated_benchmarks(test_fraction=0.30, random_seed=16)

macro_curves = curves.loc[curves["target"] == "macro"].copy()
r2_summary = (
    macro_curves.groupby(
        ["target_domain", "method", "sample_size", "sample_fraction"],
        as_index=False,
    )["r2"]
    .agg(
        mean="mean",
        lower=lambda values: values.quantile(0.10),
        upper=lambda values: values.quantile(0.90),
    )
)
full_r2 = benchmarks.query("target == 'macro' and method == 'dedicated-full'")[
    ["target_domain", "r2"]
]
r2_summary.head()

,target_domain,method,sample_size,sample_fraction,mean,lower,upper
0,5-7 fingers,target-only,43,0.050175,0.882560,0.857694,0.914038
1,5-7 fingers,target-only,86,0.100350,0.952689,0.940035,0.959454
2,5-7 fingers,target-only,214,0.249708,0.972447,0.970850,0.974087
3,5-7 fingers,target-only,428,0.499417,0.977553,0.977051,0.978183
4,5-7 fingers,target-only,643,0.750292,0.979146,0.978675,0.979433


In [7]:
# %% hide input
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=study.target_domains,
    shared_yaxes=True,
)
for column, domain in enumerate(study.target_domains, start=1):
    for method in ["zero-shot", "target-only", "transfer"]:
        frame = r2_summary.loc[
            (r2_summary["target_domain"] == domain)
            & (r2_summary["method"] == method)
        ].sort_values("sample_fraction")
        fig.add_trace(
            go.Scatter(
                x=100 * frame["sample_fraction"],
                y=frame["mean"],
                mode="lines+markers",
                name=method,
                legendgroup=method,
                showlegend=column == 1,
                line={"width": 3, "color": METHOD_COLORS[method]},
                marker={"size": 8},
                error_y={
                    "type": "data",
                    "symmetric": False,
                    "array": frame["upper"] - frame["mean"],
                    "arrayminus": frame["mean"] - frame["lower"],
                    "width": 3,
                },
                customdata=frame["sample_size"],
                hovertemplate=(
                    f"<b>{method}</b><br>labels=%{{customdata}} (%{{x:.1f}}%)"
                    "<br>macro R2=%{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
    dedicated = full_r2.loc[full_r2["target_domain"] == domain, "r2"].iloc[0]
    fig.add_hline(
        y=dedicated,
        line_dash="dash",
        line_color=METHOD_COLORS["dedicated-full"],
        annotation_text=f"full dedicated R2={dedicated:.3f}",
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="labeled adaptation pool (%)")
fig.update_yaxes(title_text="held-out macro R2", range=[-0.05, 1.01], row=1, col=1)
fig.update_layout(
    title="Transfer buys the most accuracy when target labels are scarce",
    height=570,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

## 4. Quantify label efficiency instead of eyeballing curves

We report two complementary quantities:

- **same-budget gain**: transfer R2 minus target-only R2 at the same label fraction;
- **specialist gap**: full-dedicated R2 minus transfer R2.

A positive first value proves useful source knowledge. A specialist gap near zero means
the transferred model has matched the much larger dedicated fit.

In [8]:
mean_scores = (
    macro_curves.groupby(["target_domain", "method", "sample_fraction"], as_index=False)
    .agg(r2=("r2", "mean"), mape_percent=("mape_percent", "mean"))
)
comparison = (
    mean_scores.pivot(
        index=["target_domain", "sample_fraction"],
        columns="method",
        values="r2",
    )
    .reset_index()
    .merge(full_r2.rename(columns={"r2": "dedicated_full"}), on="target_domain")
)
comparison["same_budget_gain"] = comparison["transfer"] - comparison["target-only"]
comparison["specialist_gap"] = comparison["dedicated_full"] - comparison["transfer"]
comparison["labels_percent"] = (100 * comparison["sample_fraction"]).round(1)
comparison[
    [
        "target_domain",
        "labels_percent",
        "target-only",
        "transfer",
        "same_budget_gain",
        "dedicated_full",
        "specialist_gap",
    ]
].round(4)

,target_domain,labels_percent,target-only,transfer,same_budget_gain,dedicated_full,specialist_gap
0,5-7 fingers,5.0,0.8826,0.9261,0.0436,0.9799,0.0538
1,5-7 fingers,10.0,0.9527,0.9562,0.0035,0.9799,0.0237
2,5-7 fingers,25.0,0.9724,0.9728,0.0004,0.9799,0.0071
3,5-7 fingers,49.9,0.9776,0.9778,0.0002,0.9799,0.0022
4,5-7 fingers,75.0,0.9791,0.9793,0.0002,0.9799,0.0006
5,8-10 fingers,5.0,0.9100,0.9129,0.0029,0.9743,0.0614
6,8-10 fingers,10.0,0.9497,0.9516,0.0019,0.9743,0.0227
7,8-10 fingers,25.0,0.9624,0.9641,0.0016,0.9743,0.0103
8,8-10 fingers,50.0,0.9704,0.9708,0.0003,0.9743,0.0036
9,8-10 fingers,75.0,0.9729,0.9731,0.0002,0.9743,0.0012


In [9]:
# %% hide input
plot_frame = comparison.melt(
    id_vars=["target_domain", "labels_percent"],
    value_vars=["same_budget_gain", "specialist_gap"],
    var_name="quantity",
    value_name="R2 difference",
)
fig = px.bar(
    plot_frame,
    x="labels_percent",
    y="R2 difference",
    color="quantity",
    facet_col="target_domain",
    barmode="group",
    color_discrete_map={
        "same_budget_gain": "#00798C",
        "specialist_gap": "#D1495B",
    },
    labels={"labels_percent": "labeled adaptation pool (%)"},
    title="Transfer gain is largest early; the specialist gap closes with more labels",
)
fig.add_hline(y=0, line_color="#6B7280", line_width=1)
fig.for_each_annotation(lambda item: item.update(text=item.text.split("=")[-1]))
fig.update_layout(height=510, template="plotly_white")
fig.show()

## 5. Look at individual predictions

At 10% labels we compare the transfer model with the full dedicated specialist for
mutual capacitance `C(N,S)`. A useful foundation model should preserve the trend and
avoid catastrophic extrapolation before it has seen most target labels.

In [10]:
source_rows = data["domain"] == "2-4 fingers"
parity_records = []
score_records = []
for domain_index, target_domain in enumerate(study.target_domains):
    target_rows = data["domain"] == target_domain
    local = V0TransferLearningStudy(
        np.vstack(data.loc[source_rows, "embedding"]),
        data.loc[source_rows, TARGET_NAMES],
        np.vstack(data.loc[target_rows, "embedding"]),
        data.loc[target_rows, TARGET_NAMES],
        target_names=TARGET_NAMES,
        pooled_shape_size=12,
        alpha=10.0,
    )
    pool, test = local.target_split(
        test_fraction=0.30,
        random_seed=16 + 1_000 * domain_index,
    )
    selected = np.random.default_rng(1_600 + domain_index).choice(
        pool,
        size=round(0.10 * len(pool)),
        replace=False,
    )
    models = local.fit_models(selected)
    full_model = local.fit_models(pool)["target-only"]
    for method, model in [("transfer: 10%", models["transfer"]), ("dedicated: 100%", full_model)]:
        prediction = model.predict(local.target_features[test])
        scores = regression_scores(
            local.target_targets[test],
            prediction,
            TARGET_NAMES,
        ).query("target == 'macro'").iloc[0]
        score_records.append(
            {
                "target_domain": target_domain,
                "method": method,
                "macro R2": scores["r2"],
                "MAPE (%)": scores["mape_percent"],
            }
        )
        parity_records.extend(
            {
                "target_domain": target_domain,
                "method": method,
                "simulated": expected,
                "predicted": estimate,
            }
            for expected, estimate in zip(
                local.target_targets[test, 0],
                prediction[:, 0],
            )
        )

pd.DataFrame(score_records).round(3)

,target_domain,method,macro R2,MAPE (%)
0,5-7 fingers,transfer: 10%,0.966,4.080
1,5-7 fingers,dedicated: 100%,0.980,3.441
2,8-10 fingers,transfer: 10%,0.954,4.379
3,8-10 fingers,dedicated: 100%,0.974,3.274


In [11]:
# %% hide input
parity = pd.DataFrame(parity_records)
fig = make_subplots(
    rows=2,
    cols=2,
    row_titles=study.target_domains,
    column_titles=["transfer: 10%", "dedicated: 100%"],
)
for row_index, domain in enumerate(study.target_domains, start=1):
    for column_index, method in enumerate(["transfer: 10%", "dedicated: 100%"], start=1):
        frame = parity.loc[
            (parity["target_domain"] == domain) & (parity["method"] == method)
        ]
        bounds = [
            min(frame["simulated"].min(), frame["predicted"].min()),
            max(frame["simulated"].max(), frame["predicted"].max()),
        ]
        fig.add_trace(
            go.Scattergl(
                x=frame["simulated"],
                y=frame["predicted"],
                mode="markers",
                marker={
                    "size": 5,
                    "opacity": 0.55,
                    "color": "#00798C" if "transfer" in method else "#1F2937",
                },
                hovertemplate="simulated=%{x:.3f} fF<br>predicted=%{y:.3f} fF<extra></extra>",
                showlegend=False,
            ),
            row=row_index,
            col=column_index,
        )
        fig.add_trace(
            go.Scatter(
                x=bounds,
                y=bounds,
                mode="lines",
                line={"dash": "dash", "color": "#9CA3AF"},
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row_index,
            col=column_index,
        )
fig.update_xaxes(title_text="simulated C(N,S) (fF)")
fig.update_yaxes(title_text="predicted C(N,S) (fF)")
fig.update_layout(
    title="Ten-percent transfer already recovers the physical trend",
    height=790,
    template="plotly_white",
)
fig.show()

## 6. Claim audit

This controlled experiment supports a precise statement:

- At equal label budgets, the largest measured gain appears at 5% in the nearer
  `5-7` domain. Gains remain positive but smaller for the farther high-count domain.
- Around 25-50%, transfer approaches the full dedicated R2 while using far fewer labels.
- Zero-shot degrades as finger-count distance grows. Embeddings make transfer possible;
  they do not remove domain shift.
- At high label fractions, target-only and transfer converge, as they should.

The foundation value is therefore **label efficiency plus a shared representation**,
not a promise that source weights solve every target domain without calibration.